# Encoder failures versus Elevation and Azimuth
- Updated on 2025.12.02 by bquint

* Looks for failures/faults that indicate that we have encoder reading problems
* For each of these events, query the elevation and azimuth position
* Create a histogram to show where we have the most frequent events.

## Associated Tickets
- [SITCOM-2326](https://ls.st/sitcom-2326)

## Setup Notebook

In [ ]:
day_obs_start = 20251128
day_obs_end = 20251201
time_window_for_telemetry = "10s"

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd

from astropy.time import Time, TimeDelta
from bokeh.io import output_file
from bokeh.layouts import row
from bokeh.models import HoverTool
from bokeh.plotting import figure, show, output_notebook, save

from lsst.summit.utils.efdUtils import (
    getDayObsEndTime, 
    getDayObsStartTime, 
    getEfdData, 
    makeEfdClient
)

In [ ]:
efd_client = makeEfdClient()

start_time = getDayObsStartTime(day_obs_start)
end_time = getDayObsEndTime(day_obs_end)

## Find failure messages

When I started this analysis, I did not have idea about what messages to look for.  
I am keeping the code below to remind myself why I picked the codes below.

In [ ]:
df_warning = getEfdData(
    efd_client,
    topic="lsst.sal.MTMount.logevent_warning",
    columns="*",
    begin=start_time,
    end=end_time
)

In [ ]:
df_warning_subset = df_warning.drop_duplicates(subset="code")

for idx, row in df_warning_subset.iterrows():
    if "EIB" in row.text:
        print(f"Error code: {row.code} at {idx}\n{row.text}\n")
    elif "encoder" in row.text:
        print(f"Error code: {row.code} at {idx}\n{row.text}\n")

<br><hr>
The codes we are interested in are `714`, `715`, `716`, and `717`.

In [ ]:
print(f"Dataframe size prior to filtering: {df_warning.index.size}")
mask = df_warning.code.isin([714, 715, 716, 717])
df_warning = df_warning[mask]
print(f"Dataframe size after to filtering: {df_warning.index.size}")

Cool.  
Now we can start extracting the telemetry for each failure or warning message. 

## Query Az/El Telemetry

In [ ]:
azel_data = []

for idx, row in df_warning.iterrows():

    el = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTMount.elevation",
        columns="actualPosition",
        begin=Time(idx) - TimeDelta("10s"),
        end=Time(idx),
        warn=False
    )

    az = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTMount.azimuth",
        columns="actualPosition",
        begin=Time(idx) - TimeDelta("10s"),
        end=Time(idx),
        warn=False
    )

    try:
        azel_data.append([
            idx,
            az['actualPosition'].mean(), 
            el['actualPosition'].mean()
        ])
    # sometimes the query returns an empty dataframe, let's just ignore those
    except KeyError: 
        continue


df = pd.DataFrame(data=azel_data, columns=["timestamp", "azimuth", "elevation"])

## Plot histograms

Matplotlib is easier to record a PNG file.  
The bokeh version below is useful is you want more interactivity.

In [ ]:
fig, (ax1, ax2) = plt.subplots(num="histograms", ncols=2, figsize=(12, 5))

ax1.hist(df['azimuth'], bins=360//5, fc="C0", ec="white")
ax1.set_xlabel("Azimuth [deg]")
ax1.set_ylabel("N-Faults")
ax1.grid(":", alpha=0.2)

ax2.hist(df['elevation'], bins=90//5, fc="C1", ec="white")
ax2.set_xlabel("Elevation [deg]")
ax2.set_ylabel("N-Faults")
ax2.grid(":", alpha=0.2)

fig.suptitle(f"Encoder Error Frequency at different angles\nStart: {start_time}, end: {end_time}")
fig.tight_layout()
plt.savefig(f"encoder_faults_vs_azel_{day_obs_start}_{day_obs_end}.png")
plt.show()

In [ ]:
output_notebook()

# Compute histogram data manually for azimuth
az_hist, az_edges = np.histogram(df['azimuth'], bins=360//5)
az_centers = (az_edges[:-1] + az_edges[1:]) / 2
az_width = az_edges[1] - az_edges[0]

# Compute histogram data manually for elevation
el_hist, el_edges = np.histogram(df['elevation'], bins=90//5)
el_centers = (el_edges[:-1] + el_edges[1:]) / 2
el_width = el_edges[1] - el_edges[0]

# Create azimuth plot
p1 = figure(
    width=550, 
    height=400,
    title="Azimuth",
    x_axis_label="Azimuth [deg]",
    y_axis_label="N-Faults",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

az_source = {'center': az_centers, 'count': az_hist, 'width': [az_width]*len(az_centers)}
p1.vbar(
    x='center', 
    top='count', 
    width='width', 
    source=az_source,
    color='#1f77b4',
    line_color='white'
)

hover1 = HoverTool(tooltips=[
    ('Azimuth', '@center{0.1f}°'),
    ('N-Faults', '@count')
])
p1.add_tools(hover1)
p1.grid.grid_line_alpha = 0.3

# Create elevation plot
p2 = figure(
    width=550, 
    height=400,
    title="Elevation",
    x_axis_label="Elevation [deg]",
    y_axis_label="N-Faults",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

el_source = {'center': el_centers, 'count': el_hist, 'width': [el_width]*len(el_centers)}
p2.vbar(
    x='center', 
    top='count', 
    width='width', 
    source=el_source,
    color='#ff7f0e',
    line_color='white'
)

hover2 = HoverTool(tooltips=[
    ('Elevation', '@center{0.1f}°'),
    ('N-Faults', '@count')
])
p2.add_tools(hover2)
p2.grid.grid_line_alpha = 0.3

# Combine plots and show
layout = row(p1, p2)
show(layout)

# Optional: save to HTML
output_file(f"encoder_faults_vs_azel_{day_obs_start}_{day_obs_end}.html")
save(layout)